In [52]:
# dotenv initialization
import dotenv
import os

dotenv.load_dotenv()

True

In [53]:
import pandas as pd
from langchain_core.documents import Document

data = pd.read_csv("dataset/Tamil_movies_dataset.csv")
df = pd.read_csv("dataset/tamil.csv") 

def generate_unified_profile(row):
    # Mapping logic for different headers
    name = row.get('MovieName') or row.get('Title')
    genre = row.get('Genre')
    director = row.get('Director')
    actor = row.get('Actor') or row.get('Cast')
    year = row.get('Year') or row.get('Release Year')
    rating = row.get('Rating')
    plot = row.get('Plot', 'No plot available')

    return (
        f"Title: {name}\n"
        f"Genre: {genre}\n"
        f"Director: {director}\n"
        f"Actor: {actor}\n"
        f"Release Year: {year}\n"
        f"Rating: {rating}\n"
        f"Synopsis: {plot}\n"
    )

data["profile"] = data.apply(generate_unified_profile, axis=1)
docs_1 = [
    Document(
        page_content=row["profile"],
        metadata={
            "source": "Tamil_movies_dataset",
            "title": row.get('MovieName'),
            "genre": row.get('Genre'),
            "year": row.get('Year')
        }
    ) for _, row in data.iterrows()
]

df["profile"] = df.apply(generate_unified_profile, axis=1)
docs_2 = [
    Document(
        page_content=row["profile"],
        metadata={
            "source": "Tamil_dataset_2",
            "title": row.get('Title'),
            "genre": row.get('Genre'),
            "year": row.get('Release Year')
        }
    ) for _, row in df.iterrows()
]

documents = docs_1 + docs_2

print(f"Total documents prepared for TrailerCraft: {len(documents)}")


Total documents prepared for TrailerCraft: 745


In [54]:
# Embeddings 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=Chroma.from_documents(documents,embeddings,persist_directory="dataset/Tamil_movies_dataset_chroma")

In [55]:
# Output Classes
from pydantic import BaseModel,Field
from typing import List

class Shot(BaseModel):
    shotnumber:int
    visual:str=Field(description="A brief description of the visual content of the shot.")
    cameraangle:str=Field(description="The camera angle used in the shot, e.g., close-up, wide shot, aerial view.")
    audio_cue:str=Field(description="Any significant audio cues present in the shot, such as dialogue, sound effects, or music.")

class TrailerPackage(BaseModel):
    structure: str = Field(description="The 3-act breakdown of the trailer")
    voice_over: str = Field(description="The script for the narrator it should be in tamil and that tamil is not pure it should be a tamil in a way that it is used in common")
    music_mood: str = Field(description="Instrumentation, tempo, and vibe")
    fonrstyle: str = Field(description="The font style to be used in the trailer")
    title: str = Field(description="The title of the movie in tamil and english that is good make it catchy and appealing")
    shot_list: List[Shot]


In [56]:
# Prompt template for the model
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate

retrivar = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt =ChatPromptTemplate.from_template(
    """
    You are a Kollywood Trailer Editor who is an expert in creating engaging and captivating trailers for Tamil movies.
    Your task is to analyze the provided movie plot and generate a detailed trailer structure that includes a 3-act breakdown, voice-over script, music mood, and a shot list with descriptions of visuals, camera angles, and audio cues using the synopsis of the movie
    CONTEEXT FROM DATABASE: {context}
    MOVIE SYNOPSIS: {synopsis}
    ADDITIONAL STYLE GUIDELINES: {instructions}
    """
)

In [57]:
# LLM Integration with Groq and Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.getenv("GEMINI_API_KEY"),temperature=0.7)

def generate_trailer_package(payload: dict) -> TrailerPackage:
    structeredllm=model.with_structured_output(TrailerPackage)
    synopsis = payload.get("synopsis")
    user_instructions = payload.get("user_instructions", None)
    relevant_docs = retrivar.invoke(synopsis)
    context_data = "\n\n".join([doc.page_content for doc in relevant_docs])
    chain=prompt|structeredllm
    response= chain.invoke({"synopsis": synopsis,"context": context_data,"instructions": user_instructions if user_instructions else "No specific style requested."})
    data = response.model_dump()
    
    for key, value in data.items():
        title = key.replace("_", " ").upper()
        print(f"{title}")
        if isinstance(value, list):
            for item in value:
                shot_num = item.get('shot_number', '-')
                print(f"Shot {shot_num}: {item.get('visual')}")
                print(f"Camera Angle: {item.get('cameraangle')}")
                print(f"Audio Cue: {item.get('audio_cue')}\n")
        else:
            print(f"{value}\n")
            
    return response    

In [58]:
#response from gemini
repsone=generate_trailer_package({"synopsis":"""
In the heart of the Madurai hinterlands, Bhoopathy is a respected patriarch whose influence has fueled the rise of the corrupt politician Rathnam. His dutiful son, Arjun, is sent away to the city for safety after a violent skirmish, where he reconnects with his childhood friend Anjali. While their romance blossoms amidst the urban landscape, Rathnam’s greed turns into a deadly vendetta against Bhoopathy’s lineage, prompting a clash of loyalties.
Arjun eventually returns home to defend his family’s honor, but a rift forms when Anjali demands he choose between her and his father’s dangerous legacy. Though they seemingly reconcile, a shocking assassination attempt on Bhoopathy reveals that Anjali has been acting as a mole for Rathnam to avenge her father's death. The betrayal shatters Arjun, leaving him caught between his love for a spy and his devotion to a father who might be innocent.
""","user_instructions":"Make it in Hari style"})  

STRUCTURE
Act 1: The World of Madurai & Budding Romance (0:00-0:45) - Introduces the powerful patriarch Bhoopathy and his dutiful son Arjun in the traditional Madurai setting. Highlights Arjun's simple life and his blossoming romance with Anjali. Hints at the underlying political tension with Rathnam and an initial skirmish that sends Arjun to the city. Mood is set with a mix of rustic charm and subtle foreboding. Act 2: Escalation & The Shocking Betrayal (0:45-1:30) - Rathnam's vendetta against Bhoopathy's lineage intensifies, forcing Arjun's return to defend his family. Shows glimpses of high-stakes confrontations and Arjun's commitment to his father. The romance with Anjali faces a challenge, seemingly reconciles, only to culminate in the shocking assassination attempt on Bhoopathy and the reveal of Anjali as Rathnam's mole. This act is characterized by rising tension, emotional turmoil, and a dramatic twist. Act 3: The Son's Vengeance & Moral Dilemma (1:30-2:15) - Arjun is shattere